# Repair missing Colab jobs and extract embeddings

                Este notebook va dentro de `google_drive_full_finetuning_pack`. Sirve para reparar jobs faltantes con GPU y extraer embeddings multicapa desde checkpoints. Es reanudable:

                - el entrenamiento guarda `last.pt`, `best.pt`, `history.csv` y `progress.json`;
                - si Colab se corta, vuelve a ejecutar con el mismo `DRIVE_PACK_DIR`;
                - si `done.json` existe, el job se salta;
                - la extraccion de embeddings salta perfiles ya completos salvo que uses `FORCE_EXTRACT=True`.

In [ ]:
from pathlib import Path
import subprocess, sys, os, json

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as exc:
    print("No estoy en Colab o Drive ya esta montado:", exc)

# AJUSTA ESTA RUTA si el pack esta en otra carpeta de Drive.
DRIVE_PACK_DIR = Path("/content/drive/MyDrive/google_drive_full_finetuning_pack")
assert DRIVE_PACK_DIR.exists(), f"No existe DRIVE_PACK_DIR: {DRIVE_PACK_DIR}"
print("PACK:", DRIVE_PACK_DIR)

def run_cmd(cmd, cwd=DRIVE_PACK_DIR):
    print("\n$", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        [str(x) for x in cmd],
        cwd=str(cwd),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    rc = proc.wait()
    if rc != 0:
        raise subprocess.CalledProcessError(rc, cmd)

In [ ]:
# Instalacion minima. Torch/torchvision ya suelen venir en Colab.
req = DRIVE_PACK_DIR / "requirements_colab.txt"
if req.exists():
    run_cmd([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)])

In [ ]:
# Repara el job faltante conocido: brain_tumor_mri_14c + efficientnet_b0.
# Puedes cambiar DATASETS/MODELS para correr otros faltantes.
DATASETS = "brain_tumor_mri_14c"
MODELS = "efficientnet_b0"
EPOCHS = 15
MAX_JOBS = 0  # 0 = sin limite
CONFIG = DRIVE_PACK_DIR / "configs" / "repair_missing_config.json"

cmd = [
    sys.executable,
    str(DRIVE_PACK_DIR / "scripts" / "full_finetune_colab.py"),
    "run",
    "--pack-dir", str(DRIVE_PACK_DIR),
    "--config", str(CONFIG),
    "--datasets", DATASETS,
    "--models", MODELS,
    "--epochs", str(EPOCHS),
]
if MAX_JOBS:
    cmd += ["--max-jobs", str(MAX_JOBS)]
run_cmd(cmd)

In [ ]:
# Consolida resultados softmax del pack.
run_cmd([
    sys.executable,
    str(DRIVE_PACK_DIR / "scripts" / "full_finetune_colab.py"),
    "collect",
    "--pack-dir", str(DRIVE_PACK_DIR),
])

In [ ]:
# Extrae embeddings multicapa desde los checkpoints disponibles en el pack.
# Esto tambien usa GPU si esta disponible, pero puede correr en CPU si ya no tienes GPU.
OUTPUT_ROOT = DRIVE_PACK_DIR / "extracted_embeddings"
FORCE_EXTRACT = False

cmd = [
    sys.executable,
    str(DRIVE_PACK_DIR / "scripts" / "extract_full_finetuned_embeddings_pack.py"),
    "--results-root", str(DRIVE_PACK_DIR / "results"),
    "--output", str(OUTPUT_ROOT),
    "--datasets", DATASETS,
    "--models", MODELS,
    "--batch-size", "32",
    "--num-workers", "2",
]
if FORCE_EXTRACT:
    cmd.append("--force")
run_cmd(cmd)
print("Embeddings extraidos en:", OUTPUT_ROOT)

In [ ]:
# Inventario rapido de outputs para copiar de vuelta al PC si corresponde.
for p in [
    DRIVE_PACK_DIR / "results" / "ALL_FINETUNE_RESULTS.csv",
    DRIVE_PACK_DIR / "results" / "FAILED_JOBS.csv",
    DRIVE_PACK_DIR / "extracted_embeddings" / "data" / "colab_full_finetune_embedding_extracted.csv",
    DRIVE_PACK_DIR / "extracted_embeddings" / "data" / "colab_full_finetune_embedding_errors.csv",
]:
    print(p, "EXISTS" if p.exists() else "missing")